# Round 3 notebook 02 — HYDROGEL_PACK four-bot inference

        Question: we were told HYDROGEL has 4 traders. Counterparty names are redacted, so this notebook tests
        the claim from order-book and trade-process fingerprints.

        Result preview: the data strongly supports **four functional bot roles**, not four named trader IDs:
        a light symmetric inner maker, a heavy symmetric wall maker, an aggressive buyer process, and an
        aggressive seller process. The direct edge is not copy-trading named bots; it is understanding their
        payoff horizons and quoting against the taker processes.

In [1]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

from pathlib import Path
import math
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

DATA = Path("data/ROUND3")
DAYS = [0, 1, 2]

def load_round3():
    prices = []
    trades = []
    for day in DAYS:
        p = pd.read_csv(DATA / f"prices_round_3_day_{day}.csv", sep=";")
        t = pd.read_csv(DATA / f"trades_round_3_day_{day}.csv", sep=";")
        p["day"] = day
        t["day"] = day
        prices.append(p)
        trades.append(t)
    prices = pd.concat(prices, ignore_index=True)
    trades = pd.concat(trades, ignore_index=True)
    for col in prices.columns:
        if col.startswith(("bid_", "ask_")) or col == "mid_price":
            prices[col] = pd.to_numeric(prices[col], errors="coerce")
    trades["price"] = pd.to_numeric(trades["price"], errors="coerce")
    trades["quantity"] = pd.to_numeric(trades["quantity"], errors="coerce")
    return prices, trades

def add_book_features(prices):
    p = prices.copy()
    p["best_bid"] = p["bid_price_1"]
    p["best_ask"] = p["ask_price_1"]
    p["spread"] = p["best_ask"] - p["best_bid"]
    bid_cols = ["bid_price_1", "bid_price_2", "bid_price_3"]
    ask_cols = ["ask_price_1", "ask_price_2", "ask_price_3"]
    p["wall_bid"] = p[bid_cols].min(axis=1)
    p["wall_ask"] = p[ask_cols].max(axis=1)
    p["wall_mid"] = (p["wall_bid"] + p["wall_ask"]) / 2
    p["wall_spread"] = p["wall_ask"] - p["wall_bid"]
    p["n_bid_levels"] = p[bid_cols].notna().sum(axis=1)
    p["n_ask_levels"] = p[ask_cols].notna().sum(axis=1)
    p["mid_minus_wall"] = p["mid_price"] - p["wall_mid"]
    return p

def add_future_mid(prices, horizons=(1, 5, 10, 25, 50, 100, 200, 500, 1000, 2000, 5000)):
    p = prices.sort_values(["product", "day", "timestamp"]).copy()
    for h in horizons:
        p[f"fut_mid_{h}"] = p.groupby(["product", "day"])["mid_price"].shift(-h)
        p[f"ret_{h}"] = p[f"fut_mid_{h}"] - p["mid_price"]
    return p

def attach_trades(prices, trades):
    t = trades.merge(
        prices,
        left_on=["day", "timestamp", "symbol"],
        right_on=["day", "timestamp", "product"],
        how="left",
    )
    t["side"] = np.where(
        t["price"] == t["ask_price_1"],
        "buy_agg",
        np.where(t["price"] == t["bid_price_1"], "sell_agg", "other"),
    )
    return t

def maker_markout_table(trades_with_book, horizons=(1, 10, 50, 100, 200, 500, 1000, 2000, 5000)):
    t = trades_with_book.copy()
    for h in horizons:
        # PnL for a passive maker who sold to an aggressive buyer, or bought from an aggressive seller.
        t[f"mk_{h}"] = np.where(
            t["side"] == "buy_agg",
            t["price"] - t[f"fut_mid_{h}"],
            np.where(t["side"] == "sell_agg", t[f"fut_mid_{h}"] - t["price"], np.nan),
        )
    agg = {
        "n": ("price", "size"),
        "avg_qty": ("quantity", "mean"),
        "avg_price": ("price", "mean"),
    }
    for h in horizons:
        agg[f"mk_{h}"] = (f"mk_{h}", "mean")
    return t.groupby(["symbol", "side"]).agg(**agg).round(3).reset_index()

def slow_ema_payoff(prices, span=1000, horizons=(10, 25, 50, 100, 200, 500, 1000, 2000)):
    rows = []
    p = prices.sort_values(["product", "day", "timestamp"]).copy()
    alpha = 2 / (span + 1)
    p["ema"] = p.groupby(["product", "day"])["mid_price"].transform(
        lambda s: s.ewm(alpha=alpha, adjust=False).mean()
    )
    p["dev"] = p["mid_price"] - p["ema"]
    for product, g in p.groupby("product"):
        if g["mid_price"].std() == 0:
            rows.append({"product": product, "dev_q10": 0, "dev_q90": 0, "best_H": None, "edge_ticks": 0})
            continue
        lo, hi = g["dev"].quantile(0.10), g["dev"].quantile(0.90)
        low = g[g["dev"] <= lo]
        high = g[g["dev"] >= hi]
        best = None
        for h in horizons:
            low_ret = low.groupby("day")["mid_price"].shift(-h) if False else low[f"ret_{h}"]
            high_ret = high[f"ret_{h}"]
            edge = (low_ret.mean() - high_ret.mean()) / 2
            rec = (h, edge, low_ret.mean(), high_ret.mean())
            if best is None or edge > best[1]:
                best = rec
        rows.append({
            "product": product,
            "dev_q10": lo,
            "dev_q90": hi,
            "best_H": best[0],
            "edge_ticks": best[1],
            "low_future_move": best[2],
            "high_future_move": best[3],
        })
    return pd.DataFrame(rows).sort_values("edge_ticks", ascending=False).round(3)

In [2]:
prices_raw, trades_raw = load_round3()
prices = add_future_mid(add_book_features(prices_raw))
trades = attach_trades(prices, trades_raw)
h = prices[prices["product"] == "HYDROGEL_PACK"].copy()
ht = trades[trades["symbol"] == "HYDROGEL_PACK"].copy()

display(Markdown("## HYDROGEL quote layers"))
level_rows = []
for side in ["bid", "ask"]:
    for level in [1, 2, 3]:
        px = f"{side}_price_{level}"
        vol = f"{side}_volume_{level}"
        d = h[["day", "timestamp", "mid_price", "wall_mid", px, vol]].dropna().copy()
        d = d.rename(columns={px: "price", vol: "volume"})
        d["side"] = side
        d["level"] = level
        d["abs_volume"] = d["volume"].abs()
        d["distance_from_mid"] = np.where(side == "bid", d["mid_price"] - d["price"], d["price"] - d["mid_price"])
        d["distance_from_wall_mid"] = np.where(side == "bid", d["wall_mid"] - d["price"], d["price"] - d["wall_mid"])
        level_rows.append(d)
levels = pd.concat(level_rows, ignore_index=True)
layer_summary = levels.groupby(["side", "level"]).agg(
    n=("price", "size"),
    vol_min=("abs_volume", "min"),
    vol_median=("abs_volume", "median"),
    vol_max=("abs_volume", "max"),
    dist_mid_median=("distance_from_mid", "median"),
    dist_mid_p05=("distance_from_mid", lambda s: s.quantile(0.05)),
    dist_mid_p95=("distance_from_mid", lambda s: s.quantile(0.95)),
    dist_wall_median=("distance_from_wall_mid", "median"),
).round(3)
display(layer_summary)

display(Markdown("## Symmetry evidence: same-size bid/ask quote bots"))
sym_rows = []
for level in [1, 2, 3]:
    both = h[f"bid_volume_{level}"].notna() & h[f"ask_volume_{level}"].notna()
    if both.any():
        sym_rows.append({
            "level": level,
            "both_sides_present": int(both.sum()),
            "bid_ask_volume_equal_rate": float((h.loc[both, f"bid_volume_{level}"] == h.loc[both, f"ask_volume_{level}"]).mean()),
            "bid_only": int((h[f"bid_volume_{level}"].notna() & h[f"ask_volume_{level}"].isna()).sum()),
            "ask_only": int((h[f"ask_volume_{level}"].notna() & h[f"bid_volume_{level}"].isna()).sum()),
        })
display(pd.DataFrame(sym_rows).round(4))

display(Markdown("## HYDROGEL trade/taker process"))
ht["trade_dev_wall"] = ht["price"] - ht["wall_mid"]
ht["trade_dev_mid"] = ht["price"] - ht["mid_price"]
ht["ts_mod_1000"] = ht["timestamp"] % 1000
display(pd.crosstab(ht["quantity"], ht["side"]))
display(pd.crosstab(ht["ts_mod_1000"], ht["side"]))

display(Markdown("## Passive maker payoff distance against HYDROGEL takers"))
mk = maker_markout_table(ht)
display(mk)

display(Markdown("## Taker-fingerprint clusters (diagnostic, not a production model)"))
# Sign-normalized features: positive means the market taker is aligned with recent/upcoming move.
g = h.sort_values(["day", "timestamp"]).copy()
for span in [50, 200, 1000, 2000]:
    alpha = 2 / (span + 1)
    g[f"ema{span}"] = g.groupby("day")["mid_price"].transform(lambda s: s.ewm(alpha=alpha, adjust=False).mean())
for lag in [20, 100, 500, 1000]:
    g[f"past{lag}"] = g.groupby("day")["mid_price"].diff(lag)
for fut in [100, 500, 1000]:
    g[f"future{fut}"] = g.groupby("day")["mid_price"].shift(-fut) - g["mid_price"]
ct = ht[["day", "timestamp", "side", "quantity", "price"]].merge(g, on=["day", "timestamp"], how="left")
ct = ct[ct["side"].isin(["buy_agg", "sell_agg"])].copy()
ct["dir"] = np.where(ct["side"] == "buy_agg", 1, -1)
for span in [50, 200, 1000, 2000]:
    ct[f"dir_dev_ema{span}"] = ct["dir"] * (ct["price"] - ct[f"ema{span}"])
for lag in [20, 100, 500, 1000]:
    ct[f"dir_past{lag}"] = ct["dir"] * ct[f"past{lag}"]
for fut in [100, 500, 1000]:
    ct[f"dir_future{fut}"] = ct["dir"] * ct[f"future{fut}"]
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
features = ["quantity", "dir_dev_ema50", "dir_dev_ema200", "dir_dev_ema1000", "dir_dev_ema2000", "dir_past20", "dir_past100", "dir_past500", "dir_past1000"]
X = ct[features].dropna()
ct2 = ct.loc[X.index].copy()
labels = KMeans(n_clusters=4, n_init=50, random_state=7).fit_predict(StandardScaler().fit_transform(X))
ct2["cluster"] = labels
cluster_summary = ct2.groupby("cluster").agg(
    n=("price", "size"),
    buy_frac=("dir", lambda s: (s == 1).mean()),
    avg_qty=("quantity", "mean"),
    dev50=("dir_dev_ema50", "mean"),
    dev1000=("dir_dev_ema1000", "mean"),
    prior100=("dir_past100", "mean"),
    prior1000=("dir_past1000", "mean"),
    future100=("dir_future100", "mean"),
    future500=("dir_future500", "mean"),
    future1000=("dir_future1000", "mean"),
).round(3)
display(cluster_summary)
display(pd.crosstab(ct2["cluster"], ct2["day"]))

## HYDROGEL quote layers

n  vol_min  vol_median  vol_max  dist_mid_median  \
side level                                                         
ask  1      30000      4.0        12.0     15.0              8.0   
     2      30000     10.0        25.0     30.0             10.5   
     3        516     20.0        25.0     30.0             14.5   
bid  1      30000      4.0        12.0     15.0              8.0   
     2      30000     10.0        25.0     30.0             10.5   
     3        474     20.0        25.0     30.0             14.0   

            dist_mid_p05  dist_mid_p95  dist_wall_median  
side level                                                
ask  1               7.5           8.0               8.0  
     2              10.0          11.0              10.5  
     3              14.5          15.0              10.5  
bid  1               7.5           8.0               8.0  
     2              10.0          11.0              10.5  
     3              14.0          14.5              10.5

## Symmetry evidence: same-size bid/ask quote bots

,level,both_sides_present,bid_ask_volume_equal_rate,bid_only,ask_only
0,1,30000,0.968,0,0
1,2,30000,0.967,0,0


## HYDROGEL trade/taker process

...

## Passive maker payoff distance against HYDROGEL takers

,symbol,side,n,avg_qty,avg_price,mk_1,mk_10,mk_50,mk_100,mk_200,mk_500,mk_1000,mk_2000,mk_5000
0,HYDROGEL_PACK,buy_agg,524,4.042,9997.082,7.903,7.832,8.006,7.274,6.372,4.427,2.909,-1.186,-33.090
1,HYDROGEL_PACK,sell_agg,486,4.033,9983.374,7.941,7.891,8.646,8.282,8.977,8.825,12.336,16.599,47.081


## Taker-fingerprint clusters (diagnostic, not a production model)

,n,buy_frac,avg_qty,dev50,dev1000,prior100,prior1000,future100,future500,future1000
cluster,,,,,,,,,,
0,270,0.581,4.022,4.452,8.169,-5.435,9.020,1.255,3.549,0.553
1,222,0.554,4.032,13.114,33.568,15.477,38.669,-6.573,-15.763,-23.369
2,209,0.469,4.077,2.318,-19.592,-17.610,-44.141,4.306,14.034,17.539
3,226,0.456,4.106,13.117,4.695,8.321,-15.595,1.188,3.708,5.612


day,0,1,2
cluster,,,
0,89,86,95
1,69,93,60
2,65,82,62
3,77,75,74


## HYDROGEL interpretation

        Evidence for four functional bot roles:

        - **Inner symmetric maker**: level-1 quotes are present 100% of the time, size mostly 10–15, exactly centered on `mid_price`.
        - **Wall symmetric maker**: level-2 quotes are present 100% of the time, size mostly 20–30, centered on `wall_mid`.
        - **Aggressive buyer process**: prints at `ask_price_1`, quantity 2–6, roughly balanced cadence.
        - **Aggressive seller process**: prints at `bid_price_1`, quantity 2–6, roughly balanced cadence.

        What is *not* supported:

        - There is no named-trader copy edge because buyer/seller IDs are empty.
        - The taker process is not clearly informed from side/quantity alone. Clusters are regime-like, not stable named trader IDs.

        Practical edge:

        - HYDROGEL takers pay the L1 spread. Passive maker fill quality is strongly positive for roughly 50–1000 ticks.
        - The payoff distance decays after longer horizons, so a next strategy should recycle inventory rather than carry it passively all day.
        - If we tune HYDROGEL further, the right target is fill-quality and inventory recycling around the two maker layers, not a complex predictor.